# Go2 Terrain-Adaptive Locomotion — PPO Training
Trains a blind PPO policy to traverse flat ground, slopes, and steps using proprioception only.
WBC safety layer added at inference time on your laptop.

In [ ]:
# Install dependencies
!pip install mujoco stable-baselines3[extra] gymnasium -q

In [ ]:
# Clone your repo
!git clone https://github.com/Aryaman22102002/go2-convex-mpc.git
%cd go2-convex-mpc

In [ ]:
# Copy training scripts to current directory
import shutil
shutil.copy('examples/go2_terrain_env.py', 'go2_terrain_env.py')
shutil.copy('examples/train_ppo.py', 'train_ppo.py')

# Check GPU
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Quick environment sanity check
import sys
sys.path.insert(0, '.')
from go2_terrain_env import Go2TerrainEnv

xml_path = 'models/MJCF/go2/scene.xml'
env = Go2TerrainEnv(xml_path=xml_path)
obs, _ = env.reset()
print(f'Obs shape: {obs.shape}')
print(f'Action space: {env.action_space}')

# Run a few random steps
for _ in range(10):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
print(f'Reward: {reward:.3f}  Terminated: {terminated}')
print('Environment OK')

In [ ]:
# Train -- adjust n_envs based on Colab CPU count
import multiprocessing
n_cpus = multiprocessing.cpu_count()
print(f'Available CPUs: {n_cpus}')
n_envs = min(n_cpus, 8)

!python train_ppo.py \
    --steps 5000000 \
    --n_envs {n_envs} \
    --xml models/MJCF/go2/scene.xml \
    --out go2_terrain_policy

In [ ]:
# Download the trained policy
from google.colab import files
files.download('go2_terrain_policy.zip')
files.download('checkpoints/best/best_model.zip')